🐱 Cat Detector 9000™
### A Neural Network That Finally Understands Cats Better Than I Do

> **"Is it a cat?"** — Humanity's most important unanswered question.

**Architecture:** `Image → Flatten → Hidden (ReLU) → Hidden (ReLU) → Sigmoid → 🐱 / 🚫`


In [ ]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import io
from google.colab import files
try:
    import h5py
except ImportError:
    !pip install -q h5py
    import h5py


In [ ]:
# loading dataset

print("loading training dataset")
uploaded=files.upload()

print("loading test dataset")
uploaded2=files.upload()

loading training dataset


Saving train_catvnoncat.h5 to train_catvnoncat.h5
loading test dataset


Saving test_catvnoncat.h5 to test_catvnoncat.h5


In [ ]:
with h5py.File("train_catvnoncat.h5", "r") as train_dataset:
    train_x_orig = np.array(train_dataset["train_set_x"][:])
    train_y = np.array(train_dataset["train_set_y"][:]).reshape(1, -1)
    classes = np.array(train_dataset["list_classes"][:])


# load test data
with h5py.File("test_catvnoncat.h5", 'r') as test_dataset:
  test_x_orig = np.array(test_dataset["test_set_x"][:])
  test_y=np.array(test_dataset["test_set_y"][:])

print("datasets loaded sucessfully")
print("train X shape", train_x_orig.shape)
print("Train Y:", train_y.shape)
print("Test  X:", test_x_orig.shape)
print("Test  Y:", test_y.shape)
print("Classes:", [c.decode() for c in classes])
print("Cat ratio:", f"{train_y.mean():.1%}")

datasets loaded sucessfully
train X shape (209, 64, 64, 3)
Train Y: (1, 209)
Test  X: (50, 64, 64, 3)
Test  Y: (50,)
Classes: ['non-cat', 'cat']
Cat ratio: 34.4%


In [ ]:
# prepprocessing
m_train=train_x_orig.shape[0]
m_test=test_x_orig.shape[0]
train_x= train_x_orig.reshape(m_train, -1).T/255.0
test_x=test_x_orig.reshape(m_test, -1).T/255.0


In [ ]:
print(f'train_x: {train_x.shape}  (12288 = 64x64x3 features)')
print(f'test_x : {test_x.shape}')
print(f'Pixel range: [{train_x.min():.2f}, {train_x.max():.2f}]')

train_x: (12288, 209)  (12288 = 64x64x3 features)
test_x : (12288, 50)
Pixel range: [0.00, 1.00]


In [ ]:
# neural network (pure numpy)

def sigmoid(z):
  return 1/(1+np.exp(-np.clip(z, -500, 500)))
def relu(z):
  return np.maximum(0,z)


def relu_backward(dA, z):
  dz= dA.copy()
  dz[z<=0]=0
  return dz

def init_params(dims):
    np.random.seed(42)
    p={}
    for l in range(1, len(dims)):
      p[f'W{1}']= np.random.randn(dims[1], dims[l-1])*np.sqrt(2/dims[l-1])
      p[f'b{1}']= np.zeros((dims[l],1))
      return p

def forward(X,params):
  cache={'A0':X}
  L=len(params)//2
  A=X
  for l in range(1,L):
    z=params[f'W{1}']@A + params[f'b{1}']
    A=relu(z)
    cache[f'z{l}']=z
    cache[f'A{1}']=A
  z= params[f'W{L}']@A + params[f'b{L}']
  A = sigmoid(z)

